## 2.4 图像预处理 - 从目录读取图像到组成 Batch 的完整流程

#### 1. 目标

##### 1.1 这一节的目标
前面我们已经分别学习了：
* 图像的基本概念
* NumPy / Tensor / PIL Image 的关系
* 常见图像增强 API
* Compose
* 训练集和测试集的 transforms 设计

但是这些知识点如果是分散的，到了真正做 CNN 数据准备时，还是可能会混乱。

所以这一节我们要把它们完整串起来，形成一条真正可运行的数据流。

也就是这五步：
* 读取图像组成数据集
* 拆分数据集
* 构建 Compose
* 应用 Compose
* 组成 batch

##### 1.2 这一节最重要的价值
学完这一节后，你应该能真正理解：
* 一张张磁盘里的图片，怎么变成模型能吃的数据
* 为什么中间会用到 PIL.Image
* 为什么 DataLoader 可以直接接收我们整理好的数据集
* 为什么 batch 不需要我们手动去 unsqueeze 增加 batch 维度

#### 2. 当前的数据目录结构

##### 2.1 目录结构
```
data/
    MildDemented/
    ModerateDemented/
    NonDemented/
    VeryMildDemented/
```
每个文件夹下面再放属于该类别的图片。

例如：
```
data/
    MildDemented/
        img1.jpg
        img2.jpg
        ...
    ModerateDemented/
        img1.jpg
        img2.jpg
        ...
    NonDemented/
        img1.jpg
        img2.jpg
        ...
    VeryMildDemented/
        img1.jpg
        img2.jpg
        ...
```

##### 2.2 这个目录结构为什么非常适合图像分类
因为图像分类任务最常见的组织方式就是：
* 一个文件夹代表一个类别
* 文件夹中的每张图片都属于这个类别

所以：
* MildDemented/xxx.jpg 的标签就是 MildDemented
* NonDemented/xxx.jpg 的标签就是 NonDemented

📌 也就是说，类别信息已经被“写”在文件夹名里了。

#### 3. 第一步：读取图像组成数据集

##### 3.1 主要目的：
* 扫描整个目录
* 找到所有图片文件路径
* 确定每张图片属于哪个类别
* 读取图片内容
* 让这些图片和标签形成一个“可以被模型迭代读取的数据集合”

📌 所以“数据集”不是单纯一堆图片，而是：

`图片 + 标签 + 读取规则`

##### 3.2 为什么这里要重点学习 PIL Image
在 Python 图像处理中，PIL.Image 是非常重要的一种图像对象格式。

PIL 原本是 `Python Imaging Library`

现在我们常用的是它的分支 Pillow。

最常见的读取方式是：
``` python
from PIL import Image

img = Image.open("xxx.jpg")
```
这样读出来的 img，就是一个 PIL.Image 对象。

##### 3.3 PIL Image 常见 API 介绍
**（1）Image.open()**

作用：读取图片文件，返回 PIL.Image 对象。

`img = Image.open("xxx.jpg")`

**（2）img.size**

作用：查看图像尺寸，返回 (宽, 高)。

`print(img.size)`

**（3）img.mode**

作用：查看图像模式。
* RGB：彩色图
* L：灰度图
* RGBA：带透明通道的彩色图

`print(img.mode)`

**（4）img.convert()**

作用：转换图像模式。

例如统一转换成 RGB：

`img = img.convert("RGB")`

这一步很常见，因为有些图片可能不是 RGB，例如灰度图或带透明通道图。

**（5）img.resize()**

作用：直接调整图像尺寸。

`img = img.resize((224, 224))`

不过在正式深度学习流程里，我们更常通过 transforms.Resize() 或 Compose 统一做这件事。

**（6）img.show()**

作用：临时显示图片。

`img.show()`

这个方法更适合简单调试，不常用于训练流程。

##### 3.4 用 PIL Image 读取图像的基本示例

In [2]:
from PIL import Image

img = Image.open("/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data/26 (19).jpg")

print("Type of image: ", type(img))
print("Image size: ", img.size) 
print("Image mode: ", img.mode) # L 表示灰度图像，RGB 表示彩色图像

Type of image:  <class 'PIL.JpegImagePlugin.JpegImageFile'>
Image size:  (176, 208)
Image mode:  L


这里的含义是：
* type(img)：说明这是一个 PIL.Image 类型对象
* img.mode：表示图像模式，例如 RGB
* img.size：表示图像大小，格式是 (宽, 高)

⚠️ 注意：

`这里的 size 是 (W, H)，不是 (H, W)。`

这和 NumPy / Tensor 中常见的 shape 习惯不一样，要特别注意。

##### 3.5 如何使用 PIL Image 自己组织一个“图像数据集”
如果不依赖 ImageFolder，我们也可以自己手动组织。
思路大概是：
* 遍历每个类别文件夹
* 记录图片路径
* 为每个类别分配一个标签编号
* 读取图片时使用 Image.open()
* 最后把 (图片, 标签) 配对起来

##### 3.6 手动扫描目录并记录样本信息示例
* 扫描 data/ 下所有类别文件夹
* 为类别建立编号映射
* 记录每张图片的路径和标签
* 最后形成一个 samples 列表

In [4]:
import os

data_dir = "/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data"
class_names = os.listdir(data_dir)
class_to_idx = { # 
    class_name : idx for idx, class_name in enumerate(class_names)
} # 将类别名称映射到索引0，1，2，3
print("Class to index mapping: ", class_to_idx)

samples = []

for class_name in class_names:
    class_dir = os.path.join(data_dir, class_name)

    # 过滤掉非目录项
    if not os.path.isdir(class_dir):
        continue
    # 如果是目录，则继续处理
    for file_name in os.listdir(class_dir):
        file_path = os.path.join(class_dir, file_name)

        # 过滤掉非图像文件
        if file_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif')):
            label = class_to_idx[class_name]
            samples.append((file_path, label))

print("Total samples: ", len(samples))
print("First 5 samples: ", samples[:5])


Class to index mapping:  {'VeryMildDemented': 0, 'ModerateDemented': 1, 'NonDemented': 2, 'MildDemented': 3}
Total samples:  84
First 5 samples:  [('/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data/VeryMildDemented/26 (59).jpg', 0), ('/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data/VeryMildDemented/26 (44).jpg', 0), ('/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data/VeryMildDemented/26 (46).jpg', 0), ('/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data/VeryMildDemented/26 (50).jpg', 0), ('/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data/VeryMildDemented/26 (57).jpg', 0)]


#### 4. 第二步：拆分数据集

前面我们已经有：
```
samples = [
    (image_path, label),
    ...
]
```
那么拆分其实就是把这个列表分成几部分。

In [ ]:
import random 

# 打乱样本列表，增加训练的随机性
random.shuffle(samples)

# 将样本列表分成训练集和测试集，80%用于训练，20%用于测试
train_size = int(0.8 * len(samples))
train_samples = samples[:train_size]
test_samples = samples[train_size:]

##### 为什么要先打乱再拆分
如果不打乱，可能出现问题：
* 前面全是某一类
* 后面全是另一类

这样拆出来的训练集和测试集分布可能会非常不合理。

所以一般会先：

`random.shuffle(samples)`

#### 5. 第三步：构建 Compose

##### 5.1 训练集和测试集设计不同的 Compose
**训练集 Compose**

训练集的目标是：帮助模型学习得更强。

所以通常会加入随机增强，例如：
* RandomHorizontalFlip
* RandomRotation
* ColorJitter

**测试集 Compose**

测试集的目标是：稳定、公平评估模型。

所以通常只保留确定性的预处理，例如：
* Resize
* ToTensor
* Normalize

📌 所以不是所有数据都用同一个 Compose。

##### 5.2 一个训练集 Compose 示例

In [13]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        size=(224, 224),
        scale=(0.8, 1.0),
        ratio=(0.9, 1.1)
    ),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]) # 本例子是灰度图像，只有一个通道，所以均值和标准差都是单元素列表
])


##### 5.3 一个测试集 Compose 示例

In [14]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

#### 6. 第四步：应用 Compose

##### 6.1 一个单张图片应用 Compose 的示例
``` python
from PIL import Image

img = Image.open("data/NonDemented/example.jpg").convert("RGB")
img_tensor = train_transform(img)

print(type(img_tensor))
print(img_tensor.shape)
```
可能输出：
``` python
<class 'torch.Tensor'>
torch.Size([3, 224, 224])
```
这说明：
* 原本是 PIL.Image
* 经过 Compose
* 最终变成了模型更喜欢的 Tensor


##### 6.2 如何把 Compose 应用到整个数据集
这一步本质上就是：

对数据集中的每一个 (image_path, label)，在真正取出图片时都执行一次：
* Image.open(path)
* convert("RGB")
* transform(img)

所以数据集里存储的未必是“已经处理好的所有 Tensor”，

更常见的是：
* 先存路径和标签
* 真正访问某个样本时，再即时读取并应用 transform

📌 这叫做一种“按需读取”的思想，很常见，也更节省内存。

##### 6.3 手动循环应用 Compose

In [15]:
from PIL import Image

train_data = []
test_data = []

for img, label in train_samples:
    image = Image.open(img)
    image = train_transform(image)
    train_data.append((image, label))

for img, label in test_samples:
    image = Image.open(img)
    image = test_transform(image)
    test_data.append((image, label))

这里的 train_data 中保存的就是：
* 处理后的图像 Tensor
* 对应标签

当然，这种写法适合教学理解。

正式项目里更常使用 Dataset 类按需加载，而不是一次性全部读进内存。

#### 7. 第五步：组成 batch

##### 7.1 为什么还要组成 batch
因为神经网络训练时，通常不会一次只输入一张图片。

而是一次输入多张图片，组成一个 batch。

例如：
* batch size = 32
* 表示一次输入 32 张图

📌 所以模型最常接收的不是单张图 (C, H, W)，而是一批图 (N, C, H, W)。

##### 7.2 什么是 DataLoader
DataLoader 是 PyTorch 中用来：
* 按批次读取数据
* 自动打乱顺序
* 自动把多个样本拼成 batch

它相当于“数据搬运工”。
最常见写法：

In [16]:
from torch.utils.data import Dataset, DataLoader

train_dataloader = DataLoader(
    train_data,
    batch_size=7,
    shuffle=True
)
test_dataloader = DataLoader(
    test_data,
    batch_size=7,
    shuffle=False
)

# 现在 train_dataloader 和 test_dataloader 已经准备好了，可以直接用于模型的训练和评估了
images, labels = next(iter(train_dataloader))
print("Batch of images shape: ", images.shape) # 应该是 [batch_size, channels, height, width]
print("Batch of labels shape: ", labels.shape) # 应该是 [batch_size]

Batch of images shape:  torch.Size([7, 1, 224, 224])
Batch of labels shape:  torch.Size([7])


##### 7.3 DataLoader 是如何组成 batch 的
假设 train_data 中每个样本都是：

`(img_tensor, label)`

其中 `img_tensor.shape = (3, 224, 224)`

当我们写：

`train_loader = DataLoader(train_data, batch_size=4, shuffle=True)`

然后取一个 batch：
``` python
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)
```
可能输出：
``` python
torch.Size([4, 3, 224, 224])
torch.Size([4])
```
这说明：
* DataLoader 自动把 4 张图堆叠起来了
* 形成了新的第 0 维，也就是 batch 维度

##### 7.4 为什么在组成 batch 时，我们没有手动增加一个 batch 维度
batch 维度不是在单张图阶段手动加的，而是由 DataLoader 在拼接多个样本时自动加出来的。

单张图像在经过 ToTensor() 之后，形状通常是：

`(3, 224, 224)`

这只是单张图，没有 batch 维度。

当 DataLoader 收集多个样本时，它会自动把这些张量沿着新的第 0 维堆叠起来，也就是相当于做了类似：

`torch.stack([img1, img2, img3, img4], dim=0)`

于是就变成：

`(4, 3, 224, 224)`

📌 所以 batch 维度不是我们手动 unsqueeze(0) 加的，

而是 DataLoader 在“把多张图打包成一批”时自动生成的。

##### 7.5 什么时候才需要手动增加 batch 维度
如果你手里只有一张图片，想直接送进模型，而不是通过 DataLoader，
这时候就需要手动加 batch 维度。
``` python
img = Image.open("xxx.jpg").convert("RGB")
img = test_transform(img)   # (3, 224, 224)

img = img.unsqueeze(0)      # (1, 3, 224, 224)
```
所以：
* 通过 DataLoader 取 batch → 不用手动加 batch 维度
* 单张图片直接预测 → 通常要手动加 batch 维度

##### 7.6 为什么这里“现在不需要手动创建 Dataset 类”也能传给 DataLoader
DataLoader 一定只能接收我们自己写的 Dataset 子类对象？

其实不是。

DataLoader 更关心的是：

你传进去的对象是否像一个“数据集”，也就是是否满足基本的数据集协议，例如：
* 能知道总长度
* 能通过索引拿到某一个样本

比如一个这样的列表：
``` python
train_data = [
    (img_tensor1, label1),
    (img_tensor2, label2),
    ...
]
```
它本质上也是一个“索引式数据集合”：
* len(train_data) 可以得到长度
* train_data[0] 可以取出一个样本

所以对于教学小案例来说，DataLoader 是可以直接接收这种由 (图像, 标签) 组成的列表的。

例如：

`train_loader = DataLoader(train_data, batch_size=32, shuffle=True)`

这是可以工作的。

##### 7.7 那为什么正式项目里通常还是要写 Dataset 或使用现成 Dataset
因为列表方式虽然简单，但有局限：
* 可能要一次性把所有图片都读入内存
* 不够灵活
* 不方便大型数据集
* 不方便按需读取和更复杂逻辑

所以正式项目里更常见的是：
* 使用 ImageFolder
* 或自己继承 torch.utils.data.Dataset

但在当前阶段，用列表来理解流程，其实非常有帮助。

#### 8. 完整示例：手动读取 + 手动拆分 + 手动应用Compose + 直接传递给Dataloader

##### 8.1 从目录扫描到 batch 的完整示例

In [17]:
# =========================
# 1. 扫描目录，建立样本列表
# =========================
data_dir = "/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data"
class_names = os.listdir(data_dir)
class_to_idx = {
    class_name : idx for idx, class_name in enumerate(class_names)
}
print("Class to index mapping: ", class_to_idx)

for class_name in class_names:
    class_dir = os.path.join(data_dir, class_name)
    if not os.path.isdir(class_dir):
        continue
    for file_name in os.listdir(class_dir):
        file_path = os.path.join(class_dir, file_name)
        if file_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif')):
            label = class_to_idx[class_name]
            samples.append((file_path, label))
print("Total samples: ", len(samples))
print("First 5 samples: ", samples[:5])

# =========================
# 2. 打乱并拆分数据集
# =========================
train_size = int(0.8 * len(samples))
train_samples = samples[:train_size]
test_samples = samples[train_size:]
print("Training samples: ", len(train_samples))
print("Testing samples: ", len(test_samples))

# =========================
# 3. 构建 Compose
# =========================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )
])

# =========================
# 4. 应用 Compose，得到真正的数据集内容
# =========================
train_data = []
test_data = []
for img, label in train_samples:
    image = Image.open(img)
    image = train_transform(image)
    train_data.append((image, label))
for img, label in test_samples:
    image = Image.open(img)
    image = test_transform(image)
    test_data.append((image, label))
print("First transformed training sample - image shape: ", train_data[0][0].shape, "label: ", train_data[0][1])

# =========================
# 5. 使用 DataLoader 组成 batch
# =========================
train_dataloader = DataLoader(
    train_data,
    batch_size=7,
    shuffle=True
)
test_dataloader = DataLoader(
    test_data,
    batch_size=7,
    shuffle=False
)
images, labels = next(iter(train_dataloader))
print("Batch of images shape: ", images.shape) # 应该是 [batch_size, channels, height, width]
print("Batch of labels shape: ", labels.shape) # 应该是 [batch_size]

Class to index mapping:  {'VeryMildDemented': 0, 'ModerateDemented': 1, 'NonDemented': 2, 'MildDemented': 3}
Total samples:  168
First 5 samples:  [('/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data/NonDemented/26 (78).jpg', 2), ('/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data/MildDemented/26 (23).jpg', 3), ('/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data/MildDemented/27 (7).jpg', 3), ('/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data/ModerateDemented/32.jpg', 1), ('/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data/ModerateDemented/31.jpg', 1)]
Training samples:  134
Testing samples:  34
First transformed training sample - image shape:  torch.Size([1, 224, 224]) label:  2
Batch of images shape:  torch.Size([7, 1, 224, 224])
Batch of labels shape:  torch.Size([7])


##### 8.2 流程：
1. 第一步：扫描目录
    * 得到所有图片路径和标签。
2. 第二步：拆分数据集
    * 分成训练集和测试集。
3. 第三步：构建 Compose
    * 分别定义训练和测试的预处理流程。
4. 第四步：应用 Compose
    * 真正读取图片，并转换成 Tensor。
5. 第五步：组成 batch
    * 把多张图片自动拼接成模型输入格式。

##### 8.3 数据形态变化：
1. 磁盘中的原始状态
2. 用PIL读取之后：
    * PIL.Image 类型
3. 经过 Compose之后
    * torch.Tensor # # 形状通常是 (C, H, W)
4.  经过 DataLoader 组成 batch 后
    * torch.Tensor  # 形状变成 (N, C, H, W)

#### 9. 使用 ImageFolder优化

##### 9.1 我们之前的方式是“纯手动流程”
也就是每一步都自己做：
1. 手动扫描目录
    * 自己用 os.listdir()、os.path.join() 去遍历每个类别文件夹。
2. 手动生成样本列表
    * 自己整理出：samples = [(image_path, label), …]
3. 手动应用 Compose
    * 手动在循环中，Image.open(path) + train_transform(img)
4. 手动把结果整理成可以交给 DataLoader 的形式
    * train_data = [(img_tensor, label), ...]
5. 再交给 DataLoader 组成 batch

📌 这种方式的优点是：

非常适合学习底层流程。

你能清楚看到每一步到底发生了什么。

##### 9.2 如果使用 ImageFolder，很多步骤会被自动封装
ImageFolder 会帮我们自动完成“读取目录并生成数据集”这部分工作。
**(1)自动读取目录**

只要目录结构是：
```
data/
    class1/
    class2/
    class3/
```    
`ImageFolder(root="data") `就会自动：
* 扫描所有类别文件夹
* 自动建立 class_to_idx
* 自动记录每张图片路径和标签

所以你不需要再手动写：
``` python
os.listdir()
samples.append((file_path, label))
```

**(2)可以在创建 ImageFolder 时直接传入 transform**

例如：
``` python
from torchvision import datasets

dataset = datasets.ImageFolder(
    root="data",
    transform=train_transform
)
```

这表示：
* 当 ImageFolder 取某张图片时
* 会自动用 PIL.Image.open() 读取
* 再自动应用 train_transform

所以这里也不需要你手动写：
``` python
img = Image.open(path).convert("RGB")
img = train_transform(img)
```

📌 也就是说，Compose 可以在 ImageFolder 里“自动接上”。

**(3)数据集仍然可以拆分**

ImageFolder 创建出来的是一个完整数据集。

后面你仍然可以拆分，例如：
* 用 random_split
* 或者先生成索引，再用 Subset

所以流程变成：
* ImageFolder 先自动读取完整数据集
* 然后我们再拆成训练集、测试集

**(4)最后还是交给 DataLoader**

这一点没有变。

无论你是：
* 手动列表方式
* 自定义 Dataset
* ImageFolder

最后只要它是一个合法的数据集对象，都可以交给：

`DataLoader(dataset, batch_size=32, shuffle=True)`

然后由 DataLoader 自动组成 batch。

##### 9.3 按需加载思想
在调用 ImageFolder 时，是把 Compose 传进去；真正的应用发生在 ImageFolder 取样本的时候。

`dataset = ImageFolder(root="data", transform=train_transform)`

这并不是一创建 dataset 就把所有图片都立刻处理完了，

而是：
* ImageFolder 先记住这个 transform
* 以后每次取某张图片时，再执行这个 transform

#### 10. 正式案例：使用 ImageFolder + Compose + random_split / Subset + DataLoader

In [ ]:
import random
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader

# =========================
# 1. 定义训练集和测试集的 Compose
# =========================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# =========================
# 2. 分别创建带不同 transform 的 ImageFolder
# =========================
train_full = datasets.ImageFolder(root=data_dir, transform=train_transform)
test_full = datasets.ImageFolder(root=data_dir, transform=test_transform)

# =========================
# 3. 先创建一个不带 transform 的完整数据集
# =========================
full_dataset = datasets.ImageFolder(root=data_dir)
print("Total samples in full dataset: ", len(full_dataset))
print("Class name: ", full_dataset.classes)
print("Class to index mapping: ", full_dataset.class_to_idx)

# =========================
# 4. 生成索引并打乱
# =========================
indices = list(range(len(full_dataset))) # 生成一个list，包含从0到数据集长度-1的索引list
random.shuffle(indices) # 打乱索引，增加训练的随机性

train_size = int(0.8 * len(indices)) # 计算训练集大小
train_indices = indices[:train_size]
test_indices = indices[train_size:]

# =========================
# 5. 使用 Subset 构造训练集和测试集
# =========================
train_dataset = Subset(train_full, train_indices)
test_dataset = Subset(test_full, test_indices)

# =========================
# 6. 创建 DataLoader，组成 batch
# =========================
train_dataloader = DataLoader(train_dataset, batch_size=7, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=7, shuffle=False)

# =========================
# 7. 查看一个 batch
# =========================
images, labels = next(iter(train_dataloader))
print("一个 batch 的图像 shape:", images.shape)
print("一个 batch 的标签 shape:", labels.shape)


Total samples in full dataset:  84
Class name:  ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
Class to index mapping:  {'MildDemented': 0, 'ModerateDemented': 1, 'NonDemented': 2, 'VeryMildDemented': 3}
一个 batch 的图像 shape: torch.Size([7, 3, 224, 224])
一个 batch 的标签 shape: torch.Size([7])


##### 为什么创建了3 次数据集
**第一个 train_full**

是“训练版读取器”

特点：
* 同样的目录
* 但带 train_transform

**第二个 test_full**

是“测试版读取器”

特点：
* 同样的目录
* 但带 test_transform

**第三个 full_dataset**

只是一个索引参考模板

用来确定：
* 总样本数
* 类别名
* 类别映射
* train/test 索引

**Subset**
再根据同一套索引，把训练版读取器和测试版读取器切成对应部分